# 06 — Tableau Preparation

## But First, Coffee: Competitive Revenue Intelligence

### Purpose

Prepare validated analytical outputs from the preceding notebooks for an executive Tableau dashboard.

This notebook does not introduce new statistical inference. It restructures previously validated results into visualization-ready datasets covering:

1. Executive Overview
2. Competitive Pricing
3. Promotion Economics
4. Basket-Building Opportunities
5. Customer Experience
6. Commercial Access
7. Revenue Opportunities

All Tableau outputs preserve the methodological limitations established in the preceding analysis.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_CLEANED = PROJECT_ROOT / "data" / "cleaned"
TABLES = PROJECT_ROOT / "outputs" / "tables"
TABLEAU = PROJECT_ROOT / "tableau"

TABLEAU.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Tableau output:", TABLEAU)

Project root: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis
Tableau output: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis/tableau


In [3]:
# Core cleaned datasets

menu_prices = pd.read_csv(
    DATA_CLEANED / "menu_prices_cleaned.csv"
)

matched_prices = pd.read_csv(
    DATA_CLEANED / "matched_family_prices.csv"
)

reviews = pd.read_csv(
    DATA_CLEANED / "reviews_cleaned.csv"
)

channels = pd.read_csv(
    DATA_CLEANED / "channels_partnerships_cleaned.csv"
)


# Statistical outputs

statistical_summary = pd.read_csv(
    TABLES / "statistical_analysis_summary.csv"
)

research_question_status = pd.read_csv(
    TABLES / "research_question_status.csv"
)


# Revenue opportunity outputs

pricing_headroom = pd.read_csv(
    TABLES / "revenue_pricing_headroom.csv"
)

price_scenarios = pd.read_csv(
    TABLES / "revenue_price_change_scenarios.csv"
)

price_volume_scenarios = pd.read_csv(
    TABLES / "revenue_price_volume_scenarios.csv"
)

discount_scenarios = pd.read_csv(
    TABLES / "revenue_discount_scenarios.csv"
)

promotion_response = pd.read_csv(
    TABLES / "revenue_promotion_response.csv"
)

attachment_scenarios = pd.read_csv(
    TABLES / "revenue_attachment_scenarios.csv"
)

bfc_dimensions = pd.read_csv(
    TABLES / "revenue_bfc_dimension_prevalence.csv"
)

bfc_themes = pd.read_csv(
    TABLES / "revenue_bfc_theme_prevalence.csv"
)

channel_capabilities = pd.read_csv(
    TABLES / "revenue_channel_capability_matrix.csv"
)

channel_opportunities = pd.read_csv(
    TABLES / "revenue_channel_opportunity_framework.csv"
)

revenue_opportunities = pd.read_csv(
    TABLES / "revenue_opportunity_matrix.csv"
)

internal_data_roadmap = pd.read_csv(
    TABLES / "revenue_internal_data_roadmap.csv"
)

print("Validated analytical datasets loaded.")

Validated analytical datasets loaded.


In [4]:
tableau_sources = {
    "menu_prices": menu_prices,
    "matched_prices": matched_prices,
    "reviews": reviews,
    "channels": channels,
    "statistical_summary": statistical_summary,
    "research_question_status": research_question_status,
    "pricing_headroom": pricing_headroom,
    "price_scenarios": price_scenarios,
    "price_volume_scenarios": price_volume_scenarios,
    "discount_scenarios": discount_scenarios,
    "promotion_response": promotion_response,
    "attachment_scenarios": attachment_scenarios,
    "bfc_dimensions": bfc_dimensions,
    "bfc_themes": bfc_themes,
    "channel_capabilities": channel_capabilities,
    "channel_opportunities": channel_opportunities,
    "revenue_opportunities": revenue_opportunities,
    "internal_data_roadmap": internal_data_roadmap
}

for name, df in tableau_sources.items():

    print(
        f"\n{name.upper()} — {df.shape}"
    )

    print(
        df.columns.tolist()
    )


MENU_PRICES — (150, 16)
['brand', 'branch', 'category', 'product', 'regular_price_php', 'promo_price_php', 'platform', 'source_url', 'is_promo', 'effective_price_php', 'discount_php', 'discount_pct', 'category_original', 'category_standard', 'product_class', 'comparable_product_family']

MATCHED_PRICES — (15, 4)
['brand', 'comparable_product_family', 'representative_price_php', 'source_observations']

REVIEWS — (282, 52)
['brand', 'branch', 'platform', 'aggregate_rating', 'rating_volume', 'review_date', 'review_text', 'initial_theme_hint', 'source_url', 'review_text_original', 'theme_original', 'review_text_clean', 'theme_list', 'review_date_parsed', 'theme_standard_list', 'theme_add_on', 'theme_availability', 'theme_coffee_strength', 'theme_consistency', 'theme_customization', 'theme_delivery', 'theme_food_quality', 'theme_loyalty', 'theme_missing_add_on', 'theme_missing_item', 'theme_occasion', 'theme_order_accuracy', 'theme_order_identification', 'theme_packaging', 'theme_packaging

In [5]:
# BFC descriptive menu metrics

bfc_menu = (
    menu_prices[
        menu_prices["brand"]
        == "But First, Coffee"
    ]
    .copy()
)

bfc_beverages = (
    bfc_menu[
        bfc_menu["product_class"]
        == "Beverage"
    ]
    .copy()
)


# Matched-family averages

matched_avg = (
    matched_prices
    .groupby("brand")[
        "representative_price_php"
    ]
    .mean()
)

bfc_matched_avg = matched_avg[
    "But First, Coffee"
]

starbucks_matched_avg = matched_avg[
    "Starbucks"
]

cbtl_matched_avg = matched_avg[
    "The Coffee Bean & Tea Leaf"
]


# Matched-price gaps

gap_vs_starbucks_pct = (
    (
        bfc_matched_avg
        / starbucks_matched_avg
    )
    - 1
) * 100

gap_vs_cbtl_pct = (
    (
        bfc_matched_avg
        / cbtl_matched_avg
    )
    - 1
) * 100


# Promotion metrics

promo_coverage_pct = (
    bfc_menu["is_promo"]
    .mean()
    * 100
)

mean_discount_pct = (
    bfc_menu.loc[
        bfc_menu["is_promo"],
        "discount_pct"
    ]
    .mean()
)


# Tableau KPI table

executive_kpis = pd.DataFrame({
    "kpi": [
        "BFC Beverage Median Price",
        "Matched Price Gap vs Starbucks",
        "Matched Price Gap vs CBTL",
        "Observed Promotion Coverage",
        "Observed Promotion Discount",
        "Volume Lift for 20% Discount Neutrality"
    ],

    "value": [
        bfc_beverages[
            "regular_price_php"
        ].median(),

        gap_vs_starbucks_pct,

        gap_vs_cbtl_pct,

        promo_coverage_pct,

        mean_discount_pct,

        25.0
    ],

    "unit": [
        "PHP",
        "Percent",
        "Percent",
        "Percent",
        "Percent",
        "Percent"
    ],

    "display_direction": [
        "Neutral",
        "Lower",
        "Lower",
        "Observed",
        "Observed",
        "Required"
    ],

    "evidence_type": [
        "Descriptive",
        "Matched-price descriptive",
        "Matched-price descriptive",
        "Observed snapshot",
        "Observed snapshot",
        "Scenario calculation"
    ]
})

executive_kpis.round(2)

,kpi,value,unit,display_direction,evidence_type
0,BFC Beverage Median Price,130.00,PHP,Neutral,Descriptive
1,Matched Price Gap vs Starbucks,-28.50,Percent,Lower,Matched-price descriptive
2,Matched Price Gap vs CBTL,-35.06,Percent,Lower,Matched-price descriptive
3,Observed Promotion Coverage,90.00,Percent,Observed,Observed snapshot
4,Observed Promotion Discount,20.00,Percent,Observed,Observed snapshot
5,Volume Lift for 20% Discount Neutrality,25.00,Percent,Required,Scenario calculation


In [6]:
tableau_statistical_evidence = pd.DataFrame({
    "analysis": [
        "Matched Competitive Pricing"
    ],

    "test": [
        "Friedman Test"
    ],

    "test_statistic": [
        10.0
    ],

    "degrees_of_freedom": [
        2
    ],

    "p_value": [
        0.006738
    ],

    "kendall_w": [
        1.0
    ],

    "matched_families": [
        5
    ],

    "interpretation": [
        (
            "Systematic overall price-rank difference "
            "across the three brands; BFC ranked lowest "
            "in all five matched product families."
        )
    ]
})

tableau_statistical_evidence

,analysis,test,test_statistic,degrees_of_freedom,p_value,kendall_w,matched_families,interpretation
0,Matched Competitive Pricing,Friedman Test,10.0,2,0.006738,1.0,5,Systematic overall price-rank difference acros...


In [7]:
tableau_matched_pricing = (
    pricing_headroom
    .copy()
)

tableau_matched_pricing[
    "bfc_price_index_vs_starbucks"
] = (
    tableau_matched_pricing[
        "bfc_price"
    ]
    / tableau_matched_pricing[
        "starbucks_price"
    ]
    * 100
)

tableau_matched_pricing[
    "bfc_price_index_vs_cbtl"
] = (
    tableau_matched_pricing[
        "bfc_price"
    ]
    / tableau_matched_pricing[
        "cbtl_price"
    ]
    * 100
)

tableau_matched_pricing.round(2)

,product_family,bfc_price,starbucks_price,cbtl_price,gap_vs_starbucks_php,gap_vs_cbtl_php,bfc_discount_vs_starbucks_pct,bfc_discount_vs_cbtl_pct,bfc_price_index_vs_starbucks,bfc_price_index_vs_cbtl
0,Americano,130.0,175.0,185.0,45.0,55.0,25.71,29.73,74.29,70.27
1,Cafe Latte,150.0,185.0,187.5,35.0,37.5,18.92,20.00,81.08,80.00
2,Caramel Macchiato,130.0,210.0,245.0,80.0,115.0,38.10,46.94,61.90,53.06
3,Matcha Latte,130.0,190.0,225.0,60.0,95.0,31.58,42.22,68.42,57.78
4,Mocha,150.0,205.0,220.0,55.0,70.0,26.83,31.82,73.17,68.18


In [8]:
tableau_pricing_long = (
    matched_prices[
        [
            "brand",
            "comparable_product_family",
            "representative_price_php",
            "source_observations"
        ]
    ]
    .copy()
)

tableau_pricing_long = (
    tableau_pricing_long
    .rename(
        columns={
            "comparable_product_family":
                "product_family"
        }
    )
)

tableau_pricing_long[
    "price_rank"
] = (
    tableau_pricing_long
    .groupby(
        "product_family"
    )[
        "representative_price_php"
    ]
    .rank(
        method="min",
        ascending=True
    )
)

tableau_pricing_long.sort_values(
    [
        "product_family",
        "price_rank"
    ]
)

,brand,product_family,representative_price_php,source_observations,price_rank
0,"But First, Coffee",Americano,130.0,1,1.0
5,Starbucks,Americano,175.0,2,2.0
10,The Coffee Bean & Tea Leaf,Americano,185.0,2,3.0
1,"But First, Coffee",Cafe Latte,150.0,1,1.0
6,Starbucks,Cafe Latte,185.0,2,2.0
11,The Coffee Bean & Tea Leaf,Cafe Latte,187.5,2,3.0
2,"But First, Coffee",Caramel Macchiato,130.0,3,1.0
7,Starbucks,Caramel Macchiato,210.0,2,2.0
12,The Coffee Bean & Tea Leaf,Caramel Macchiato,245.0,4,3.0
3,"But First, Coffee",Matcha Latte,130.0,3,1.0


In [9]:
tableau_menu_pricing = (
    menu_prices
    .groupby(
        [
            "brand",
            "product_class"
        ]
    )
    .agg(
        menu_items=(
            "product",
            "count"
        ),

        mean_regular_price_php=(
            "regular_price_php",
            "mean"
        ),

        median_regular_price_php=(
            "regular_price_php",
            "median"
        ),

        min_regular_price_php=(
            "regular_price_php",
            "min"
        ),

        max_regular_price_php=(
            "regular_price_php",
            "max"
        )
    )
    .reset_index()
)

tableau_menu_pricing.round(2)

,brand,product_class,menu_items,mean_regular_price_php,median_regular_price_php,min_regular_price_php,max_regular_price_php
0,"But First, Coffee",Add-on,2,30.00,30.0,30,30
1,"But First, Coffee",Beverage,45,141.91,130.0,69,219
2,"But First, Coffee",Food,3,91.67,75.0,60,140
3,Starbucks,Beverage,38,196.84,195.0,140,220
4,Starbucks,Food,12,217.08,195.0,135,315
5,The Coffee Bean & Tea Leaf,Beverage,39,223.97,225.0,150,270
6,The Coffee Bean & Tea Leaf,Food,11,159.09,145.0,100,295


In [10]:
executive_kpis.to_csv(
    TABLEAU / "tableau_executive_kpis.csv",
    index=False
)

tableau_statistical_evidence.to_csv(
    TABLEAU / "tableau_statistical_evidence.csv",
    index=False
)

tableau_matched_pricing.to_csv(
    TABLEAU / "tableau_matched_pricing.csv",
    index=False
)

tableau_pricing_long.to_csv(
    TABLEAU / "tableau_pricing_long.csv",
    index=False
)

tableau_menu_pricing.to_csv(
    TABLEAU / "tableau_menu_pricing.csv",
    index=False
)

print(
    "Executive KPI and competitive pricing "
    "datasets exported successfully."
)

Executive KPI and competitive pricing datasets exported successfully.


In [11]:
print("PRICE SCENARIOS")
display(
    price_scenarios.round(2)
)

print("\nPRICE × VOLUME SCENARIOS")
display(
    price_volume_scenarios.round(2)
)

print("\nDISCOUNT SCENARIOS")
display(
    discount_scenarios.round(2)
)

print("\nPROMOTION RESPONSE")
display(
    promotion_response.round(2)
)

print("\nATTACHMENT SCENARIOS")
display(
    attachment_scenarios.round(2)
)

PRICE SCENARIOS


,price_change_pct,illustrative_avg_price_php,required_volume_index_for_revenue_neutrality,maximum_volume_decline_pct
0,0,138.00,1.00,0.00
1,3,142.14,0.97,2.91
2,5,144.90,0.95,4.76
3,7,147.66,0.93,6.54
4,10,151.80,0.91,9.09



PRICE × VOLUME SCENARIOS


,price_change_pct,volume_change_pct,revenue_index,revenue_change_pct
0,0.0,0.0,1.00,0.00
1,0.0,-2.0,0.98,-2.00
2,0.0,-5.0,0.95,-5.00
3,0.0,-8.0,0.92,-8.00
4,0.0,-10.0,0.90,-10.00
5,3.0,0.0,1.03,3.00
6,3.0,-2.0,1.01,0.94
7,3.0,-5.0,0.98,-2.15
8,3.0,-8.0,0.95,-5.24
9,3.0,-10.0,0.93,-7.30



DISCOUNT SCENARIOS


,discount_pct,price_retained_pct,required_volume_index,required_volume_increase_pct
0,5,95,1.05,5.26
1,10,90,1.11,11.11
2,15,85,1.18,17.65
3,20,80,1.25,25.00
4,25,75,1.33,33.33
5,30,70,1.43,42.86



PROMOTION RESPONSE


,volume_increase_pct,revenue_index,revenue_change_pct
0,0,0.80,-20.0
1,10,0.88,-12.0
2,20,0.96,-4.0
3,25,1.00,0.0
4,30,1.04,4.0
5,40,1.12,12.0
6,50,1.20,20.0



ATTACHMENT SCENARIOS


,attachment_rate_pct,base_transaction_value_php,expected_incremental_value_php,illustrative_atv_php,atv_increase_pct,revenue_index_if_transactions_constant,revenue_change_pct_if_transactions_constant
0,0,130,0.0,130.0,0.00,1.00,0.00
1,5,130,2.5,132.5,1.92,1.02,1.92
2,10,130,5.0,135.0,3.85,1.04,3.85
3,15,130,7.5,137.5,5.77,1.06,5.77
4,20,130,10.0,140.0,7.69,1.08,7.69
5,25,130,12.5,142.5,9.62,1.10,9.62
6,30,130,15.0,145.0,11.54,1.12,11.54


In [12]:
tableau_price_scenarios = (
    price_scenarios
    .copy()
)

tableau_price_scenarios[
    "scenario_type"
] = "Selective Pricing"

tableau_price_scenarios[
    "scenario_assumption"
] = (
    "Revenue-neutral volume response "
    "before costs"
)

tableau_price_scenarios[
    "metric"
] = "Maximum Volume Decline"

tableau_price_scenarios[
    "scenario_value_pct"
] = (
    tableau_price_scenarios[
        "price_change_pct"
    ]
)

tableau_price_scenarios[
    "outcome_value_pct"
] = (
    tableau_price_scenarios[
        "maximum_volume_decline_pct"
    ]
)

tableau_price_scenarios[
    "outcome_label"
] = "Revenue-Neutral Volume Decline"

tableau_price_scenarios[
    [
        "scenario_type",
        "scenario_value_pct",
        "outcome_value_pct",
        "outcome_label",
        "illustrative_avg_price_php",
        "scenario_assumption"
    ]
].round(2)

,scenario_type,scenario_value_pct,outcome_value_pct,outcome_label,illustrative_avg_price_php,scenario_assumption
0,Selective Pricing,0,0.00,Revenue-Neutral Volume Decline,138.00,Revenue-neutral volume response before costs
1,Selective Pricing,3,2.91,Revenue-Neutral Volume Decline,142.14,Revenue-neutral volume response before costs
2,Selective Pricing,5,4.76,Revenue-Neutral Volume Decline,144.90,Revenue-neutral volume response before costs
3,Selective Pricing,7,6.54,Revenue-Neutral Volume Decline,147.66,Revenue-neutral volume response before costs
4,Selective Pricing,10,9.09,Revenue-Neutral Volume Decline,151.80,Revenue-neutral volume response before costs


In [13]:
tableau_price_volume = (
    price_volume_scenarios
    .copy()
)

tableau_price_volume[
    "scenario_type"
] = "Price × Volume"

tableau_price_volume[
    "scenario_assumption"
] = (
    "Revenue change under combined "
    "price and volume response; before costs"
)

tableau_price_volume[
    "revenue_outcome"
] = np.select(
    [
        tableau_price_volume[
            "revenue_change_pct"
        ] > 0,

        tableau_price_volume[
            "revenue_change_pct"
        ] < 0
    ],
    [
        "Revenue Increase",
        "Revenue Decrease"
    ],
    default="Revenue Neutral"
)

tableau_price_volume.round(2)

,price_change_pct,volume_change_pct,revenue_index,revenue_change_pct,scenario_type,scenario_assumption,revenue_outcome
0,0.0,0.0,1.00,0.00,Price × Volume,Revenue change under combined price and volume...,Revenue Neutral
1,0.0,-2.0,0.98,-2.00,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
2,0.0,-5.0,0.95,-5.00,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
3,0.0,-8.0,0.92,-8.00,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
4,0.0,-10.0,0.90,-10.00,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
5,3.0,0.0,1.03,3.00,Price × Volume,Revenue change under combined price and volume...,Revenue Increase
6,3.0,-2.0,1.01,0.94,Price × Volume,Revenue change under combined price and volume...,Revenue Increase
7,3.0,-5.0,0.98,-2.15,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
8,3.0,-8.0,0.95,-5.24,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease
9,3.0,-10.0,0.93,-7.30,Price × Volume,Revenue change under combined price and volume...,Revenue Decrease


In [14]:
tableau_discount_scenarios = (
    discount_scenarios
    .copy()
)

tableau_discount_scenarios[
    "scenario_type"
] = "Discount Economics"

tableau_discount_scenarios[
    "scenario_assumption"
] = (
    "Required volume increase for "
    "revenue neutrality before costs"
)

tableau_discount_scenarios[
    "scenario_value_pct"
] = (
    tableau_discount_scenarios[
        "discount_pct"
    ]
)

tableau_discount_scenarios[
    "outcome_value_pct"
] = (
    tableau_discount_scenarios[
        "required_volume_increase_pct"
    ]
)

tableau_discount_scenarios[
    "outcome_label"
] = "Required Volume Increase"

tableau_discount_scenarios.round(2)

,discount_pct,price_retained_pct,required_volume_index,required_volume_increase_pct,scenario_type,scenario_assumption,scenario_value_pct,outcome_value_pct,outcome_label
0,5,95,1.05,5.26,Discount Economics,Required volume increase for revenue neutralit...,5,5.26,Required Volume Increase
1,10,90,1.11,11.11,Discount Economics,Required volume increase for revenue neutralit...,10,11.11,Required Volume Increase
2,15,85,1.18,17.65,Discount Economics,Required volume increase for revenue neutralit...,15,17.65,Required Volume Increase
3,20,80,1.25,25.00,Discount Economics,Required volume increase for revenue neutralit...,20,25.00,Required Volume Increase
4,25,75,1.33,33.33,Discount Economics,Required volume increase for revenue neutralit...,25,33.33,Required Volume Increase
5,30,70,1.43,42.86,Discount Economics,Required volume increase for revenue neutralit...,30,42.86,Required Volume Increase


In [15]:
tableau_promo_response = (
    promotion_response
    .copy()
)

tableau_promo_response[
    "discount_pct"
] = 20.0

tableau_promo_response[
    "scenario_type"
] = "20% Promotion Response"

tableau_promo_response[
    "scenario_assumption"
] = (
    "Illustrative revenue response to a "
    "20% discount under alternative volume lifts"
)

tableau_promo_response[
    "revenue_outcome"
] = np.select(
    [
        tableau_promo_response[
            "revenue_change_pct"
        ] > 0,

        tableau_promo_response[
            "revenue_change_pct"
        ] < 0
    ],
    [
        "Revenue Increase",
        "Revenue Decrease"
    ],
    default="Revenue Neutral"
)

tableau_promo_response.round(2)

,volume_increase_pct,revenue_index,revenue_change_pct,discount_pct,scenario_type,scenario_assumption,revenue_outcome
0,0,0.80,-20.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Decrease
1,10,0.88,-12.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Decrease
2,20,0.96,-4.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Decrease
3,25,1.00,0.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Neutral
4,30,1.04,4.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Increase
5,40,1.12,12.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Increase
6,50,1.20,20.0,20.0,20% Promotion Response,Illustrative revenue response to a 20% discoun...,Revenue Increase


In [16]:
tableau_attachment = (
    attachment_scenarios
    .copy()
)

tableau_attachment[
    "scenario_type"
] = "Basket Building"

tableau_attachment[
    "scenario_assumption"
] = (
    "Illustrative ₱130 base beverage plus "
    "₱50 incremental attached item; "
    "transactions held constant"
)

tableau_attachment[
    "scenario_value_pct"
] = (
    tableau_attachment[
        "attachment_rate_pct"
    ]
)

tableau_attachment[
    "outcome_value_pct"
] = (
    tableau_attachment[
        "atv_increase_pct"
    ]
)

tableau_attachment[
    "outcome_label"
] = "ATV Increase"

tableau_attachment.round(2)

,attachment_rate_pct,base_transaction_value_php,expected_incremental_value_php,illustrative_atv_php,atv_increase_pct,revenue_index_if_transactions_constant,revenue_change_pct_if_transactions_constant,scenario_type,scenario_assumption,scenario_value_pct,outcome_value_pct,outcome_label
0,0,130,0.0,130.0,0.00,1.00,0.00,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,0,0.00,ATV Increase
1,5,130,2.5,132.5,1.92,1.02,1.92,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,5,1.92,ATV Increase
2,10,130,5.0,135.0,3.85,1.04,3.85,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,10,3.85,ATV Increase
3,15,130,7.5,137.5,5.77,1.06,5.77,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,15,5.77,ATV Increase
4,20,130,10.0,140.0,7.69,1.08,7.69,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,20,7.69,ATV Increase
5,25,130,12.5,142.5,9.62,1.10,9.62,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,25,9.62,ATV Increase
6,30,130,15.0,145.0,11.54,1.12,11.54,Basket Building,Illustrative ₱130 base beverage plus ₱50 incre...,30,11.54,ATV Increase


In [17]:
price_long = (
    tableau_price_scenarios[
        [
            "scenario_type",
            "scenario_value_pct",
            "outcome_value_pct",
            "outcome_label",
            "scenario_assumption"
        ]
    ]
    .copy()
)

discount_long = (
    tableau_discount_scenarios[
        [
            "scenario_type",
            "scenario_value_pct",
            "outcome_value_pct",
            "outcome_label",
            "scenario_assumption"
        ]
    ]
    .copy()
)

attachment_long = (
    tableau_attachment[
        [
            "scenario_type",
            "scenario_value_pct",
            "outcome_value_pct",
            "outcome_label",
            "scenario_assumption"
        ]
    ]
    .copy()
)

tableau_revenue_scenarios = pd.concat(
    [
        price_long,
        discount_long,
        attachment_long
    ],
    ignore_index=True
)

tableau_revenue_scenarios.round(2)

,scenario_type,scenario_value_pct,outcome_value_pct,outcome_label,scenario_assumption
0,Selective Pricing,0,0.00,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs
1,Selective Pricing,3,2.91,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs
2,Selective Pricing,5,4.76,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs
3,Selective Pricing,7,6.54,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs
4,Selective Pricing,10,9.09,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs
5,Discount Economics,5,5.26,Required Volume Increase,Required volume increase for revenue neutralit...
6,Discount Economics,10,11.11,Required Volume Increase,Required volume increase for revenue neutralit...
7,Discount Economics,15,17.65,Required Volume Increase,Required volume increase for revenue neutralit...
8,Discount Economics,20,25.00,Required Volume Increase,Required volume increase for revenue neutralit...
9,Discount Economics,25,33.33,Required Volume Increase,Required volume increase for revenue neutralit...


In [18]:
input_label_map = {
    "Selective Pricing":
        "Price Increase %",
    "Discount Economics":
        "Discount %",
    "Basket Building":
        "Attachment Rate %"
}

tableau_revenue_scenarios[
    "input_label"
] = (
    tableau_revenue_scenarios[
        "scenario_type"
    ]
    .map(input_label_map)
)

tableau_revenue_scenarios[
    "evidence_type"
] = "Scenario Analysis"

tableau_revenue_scenarios[
    "profitability_status"
] = (
    "Revenue scenario only — "
    "cost and contribution margin unavailable"
)

tableau_revenue_scenarios.round(2)

,scenario_type,scenario_value_pct,outcome_value_pct,outcome_label,scenario_assumption,input_label,evidence_type,profitability_status
0,Selective Pricing,0,0.00,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs,Price Increase %,Scenario Analysis,Revenue scenario only — cost and contribution ...
1,Selective Pricing,3,2.91,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs,Price Increase %,Scenario Analysis,Revenue scenario only — cost and contribution ...
2,Selective Pricing,5,4.76,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs,Price Increase %,Scenario Analysis,Revenue scenario only — cost and contribution ...
3,Selective Pricing,7,6.54,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs,Price Increase %,Scenario Analysis,Revenue scenario only — cost and contribution ...
4,Selective Pricing,10,9.09,Revenue-Neutral Volume Decline,Revenue-neutral volume response before costs,Price Increase %,Scenario Analysis,Revenue scenario only — cost and contribution ...
5,Discount Economics,5,5.26,Required Volume Increase,Required volume increase for revenue neutralit...,Discount %,Scenario Analysis,Revenue scenario only — cost and contribution ...
6,Discount Economics,10,11.11,Required Volume Increase,Required volume increase for revenue neutralit...,Discount %,Scenario Analysis,Revenue scenario only — cost and contribution ...
7,Discount Economics,15,17.65,Required Volume Increase,Required volume increase for revenue neutralit...,Discount %,Scenario Analysis,Revenue scenario only — cost and contribution ...
8,Discount Economics,20,25.00,Required Volume Increase,Required volume increase for revenue neutralit...,Discount %,Scenario Analysis,Revenue scenario only — cost and contribution ...
9,Discount Economics,25,33.33,Required Volume Increase,Required volume increase for revenue neutralit...,Discount %,Scenario Analysis,Revenue scenario only — cost and contribution ...


In [19]:
tableau_revenue_scenarios.to_csv(
    TABLEAU / "tableau_revenue_scenarios.csv",
    index=False
)

tableau_price_volume.to_csv(
    TABLEAU / "tableau_price_volume_heatmap.csv",
    index=False
)

tableau_promo_response.to_csv(
    TABLEAU / "tableau_promotion_response.csv",
    index=False
)

print(
    "Revenue scenario datasets "
    "exported successfully."
)

Revenue scenario datasets exported successfully.


In [20]:
tableau_dimensions = (
    bfc_dimensions
    .copy()
)

tableau_dimensions[
    "evidence_level"
] = "Management Dimension"

tableau_dimensions = (
    tableau_dimensions
    .rename(
        columns={
            "dimension": "experience_factor"
        }
    )
)

tableau_dimensions[
    "brand"
] = "But First, Coffee"

tableau_dimensions[
    "interpretation"
] = (
    "Share of collected BFC reviews "
    "mentioning this management dimension"
)

tableau_dimensions[
    "metric_definition"
] = "Mention prevalence — not dissatisfaction rate"

tableau_dimensions[
    [
        "brand",
        "evidence_level",
        "experience_factor",
        "mentions",
        "prevalence_pct",
        "interpretation",
        "metric_definition"
    ]
].round(2)

,brand,evidence_level,experience_factor,mentions,prevalence_pct,interpretation,metric_definition
0,"But First, Coffee",Management Dimension,Product Quality,52,50.00,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
1,"But First, Coffee",Management Dimension,Packaging,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
2,"But First, Coffee",Management Dimension,Value & Portion,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
3,"But First, Coffee",Management Dimension,Order Execution,16,15.38,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
4,"But First, Coffee",Management Dimension,Availability & Consistency,14,13.46,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
5,"But First, Coffee",Management Dimension,Customization & Add-ons,13,12.50,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
6,"But First, Coffee",Management Dimension,Service & Delivery,8,7.69,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
7,"But First, Coffee",Management Dimension,Loyalty Experience,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate


In [21]:
tableau_themes = (
    bfc_themes
    .copy()
)

tableau_themes[
    "evidence_level"
] = "Atomic Theme"

tableau_themes = (
    tableau_themes
    .rename(
        columns={
            "theme": "experience_factor"
        }
    )
)

tableau_themes[
    "brand"
] = "But First, Coffee"

tableau_themes[
    "interpretation"
] = (
    "Share of collected BFC reviews "
    "mentioning this theme"
)

tableau_themes[
    "metric_definition"
] = "Mention prevalence — not dissatisfaction rate"

tableau_themes[
    [
        "brand",
        "evidence_level",
        "experience_factor",
        "mentions",
        "prevalence_pct",
        "interpretation",
        "metric_definition"
    ]
].round(2)

,brand,evidence_level,experience_factor,mentions,prevalence_pct,interpretation,metric_definition
0,"But First, Coffee",Atomic Theme,Taste,49,47.12,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
1,"But First, Coffee",Atomic Theme,Packaging,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
2,"But First, Coffee",Atomic Theme,Value,16,15.38,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
3,"But First, Coffee",Atomic Theme,Consistency,14,13.46,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
4,"But First, Coffee",Atomic Theme,Order Accuracy,11,10.58,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
5,"But First, Coffee",Atomic Theme,Portion,10,9.62,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
6,"But First, Coffee",Atomic Theme,Customization,9,8.65,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
7,"But First, Coffee",Atomic Theme,Service,6,5.77,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
8,"But First, Coffee",Atomic Theme,Missing Item,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate
9,"But First, Coffee",Atomic Theme,Add-on,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate


In [22]:
customer_columns = [
    "brand",
    "evidence_level",
    "experience_factor",
    "mentions",
    "prevalence_pct",
    "interpretation",
    "metric_definition"
]

tableau_customer_experience = pd.concat(
    [
        tableau_dimensions[
            customer_columns
        ],

        tableau_themes[
            customer_columns
        ]
    ],
    ignore_index=True
)

tableau_customer_experience[
    "review_base_n"
] = len(
    reviews[
        reviews["brand"]
        == "But First, Coffee"
    ]
)

tableau_customer_experience[
    "scope"
] = (
    "Collected external BFC review evidence"
)

tableau_customer_experience.sort_values(
    [
        "evidence_level",
        "prevalence_pct"
    ],
    ascending=[
        True,
        False
    ]
).round(2)

,brand,evidence_level,experience_factor,mentions,prevalence_pct,interpretation,metric_definition,review_base_n,scope
8,"But First, Coffee",Atomic Theme,Taste,49,47.12,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
9,"But First, Coffee",Atomic Theme,Packaging,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
10,"But First, Coffee",Atomic Theme,Value,16,15.38,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
11,"But First, Coffee",Atomic Theme,Consistency,14,13.46,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
12,"But First, Coffee",Atomic Theme,Order Accuracy,11,10.58,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
13,"But First, Coffee",Atomic Theme,Portion,10,9.62,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
14,"But First, Coffee",Atomic Theme,Customization,9,8.65,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
15,"But First, Coffee",Atomic Theme,Service,6,5.77,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
16,"But First, Coffee",Atomic Theme,Missing Item,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence
17,"But First, Coffee",Atomic Theme,Add-on,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence


In [23]:
revenue_mechanism_map = {
    "Product Quality":
        "Perceived value / repeat purchase",

    "Order Execution":
        "Transaction protection / service recovery",

    "Customization & Add-ons":
        "Upsell execution / incremental transaction value",

    "Packaging":
        "Delivery experience / service recovery",

    "Value & Portion":
        "Perceived value / purchase response",

    "Service & Delivery":
        "Retention / transaction experience",

    "Availability & Consistency":
        "Conversion / repeat purchase",

    "Loyalty Experience":
        "Retention / purchase frequency"
}

tableau_customer_experience[
    "potential_revenue_mechanism"
] = np.where(
    tableau_customer_experience[
        "evidence_level"
    ] == "Management Dimension",

    tableau_customer_experience[
        "experience_factor"
    ].map(
        revenue_mechanism_map
    ),

    "Descriptive theme evidence"
)

tableau_customer_experience[
    "causal_status"
] = (
    "Association / descriptive evidence only"
)

tableau_customer_experience[
    "financial_impact_status"
] = (
    "Not quantified from external data"
)

tableau_customer_experience.round(2)

,brand,evidence_level,experience_factor,mentions,prevalence_pct,interpretation,metric_definition,review_base_n,scope,potential_revenue_mechanism,causal_status,financial_impact_status
0,"But First, Coffee",Management Dimension,Product Quality,52,50.00,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Perceived value / repeat purchase,Association / descriptive evidence only,Not quantified from external data
1,"But First, Coffee",Management Dimension,Packaging,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Delivery experience / service recovery,Association / descriptive evidence only,Not quantified from external data
2,"But First, Coffee",Management Dimension,Value & Portion,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Perceived value / purchase response,Association / descriptive evidence only,Not quantified from external data
3,"But First, Coffee",Management Dimension,Order Execution,16,15.38,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Transaction protection / service recovery,Association / descriptive evidence only,Not quantified from external data
4,"But First, Coffee",Management Dimension,Availability & Consistency,14,13.46,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Conversion / repeat purchase,Association / descriptive evidence only,Not quantified from external data
5,"But First, Coffee",Management Dimension,Customization & Add-ons,13,12.50,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Upsell execution / incremental transaction value,Association / descriptive evidence only,Not quantified from external data
6,"But First, Coffee",Management Dimension,Service & Delivery,8,7.69,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Retention / transaction experience,Association / descriptive evidence only,Not quantified from external data
7,"But First, Coffee",Management Dimension,Loyalty Experience,4,3.85,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Retention / purchase frequency,Association / descriptive evidence only,Not quantified from external data
8,"But First, Coffee",Atomic Theme,Taste,49,47.12,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Descriptive theme evidence,Association / descriptive evidence only,Not quantified from external data
9,"But First, Coffee",Atomic Theme,Packaging,20,19.23,Share of collected BFC reviews mentioning this...,Mention prevalence — not dissatisfaction rate,104,Collected external BFC review evidence,Descriptive theme evidence,Association / descriptive evidence only,Not quantified from external data


In [24]:
tableau_polarity_context = pd.DataFrame({
    "polarity": [
        "Unclear",
        "Negative",
        "Positive",
        "Mixed"
    ],

    "review_count": [
        135,
        109,
        35,
        3
    ],

    "share_of_all_reviews_pct": [
        47.87,
        38.65,
        12.41,
        1.06
    ]
})

tableau_polarity_context[
    "total_reviews"
] = 282

tableau_polarity_context[
    "classification_coverage_pct"
] = 52.13

tableau_polarity_context[
    "method"
] = "Conservative rule-based polarity"

tableau_polarity_context[
    "interpretation_limit"
] = (
    "Not a population-level customer satisfaction measure"
)

tableau_polarity_context

,polarity,review_count,share_of_all_reviews_pct,total_reviews,classification_coverage_pct,method,interpretation_limit
0,Unclear,135,47.87,282,52.13,Conservative rule-based polarity,Not a population-level customer satisfaction m...
1,Negative,109,38.65,282,52.13,Conservative rule-based polarity,Not a population-level customer satisfaction m...
2,Positive,35,12.41,282,52.13,Conservative rule-based polarity,Not a population-level customer satisfaction m...
3,Mixed,3,1.06,282,52.13,Conservative rule-based polarity,Not a population-level customer satisfaction m...


In [25]:
tableau_customer_methodology = pd.DataFrame({
    "methodological_item": [
        "Review Base",
        "Theme Prevalence",
        "Multi-theme Reviews",
        "Polarity Classification",
        "Causality",
        "Financial Impact"
    ],

    "definition": [
        "104 collected BFC reviews for BFC-specific prevalence analysis",
        "Percentage of BFC reviews mentioning the theme or dimension",
        "A single review may contribute to multiple themes",
        "Conservative rule-based classification with partial coverage",
        "Theme evidence is descriptive and does not establish causality",
        "Lost revenue or contribution margin cannot be quantified externally"
    ]
})

tableau_customer_methodology

,methodological_item,definition
0,Review Base,104 collected BFC reviews for BFC-specific pre...
1,Theme Prevalence,Percentage of BFC reviews mentioning the theme...
2,Multi-theme Reviews,A single review may contribute to multiple themes
3,Polarity Classification,Conservative rule-based classification with pa...
4,Causality,Theme evidence is descriptive and does not est...
5,Financial Impact,Lost revenue or contribution margin cannot be ...


In [26]:
tableau_customer_experience.to_csv(
    TABLEAU / "tableau_customer_experience.csv",
    index=False
)

tableau_polarity_context.to_csv(
    TABLEAU / "tableau_polarity_context.csv",
    index=False
)

tableau_customer_methodology.to_csv(
    TABLEAU / "tableau_customer_methodology.csv",
    index=False
)

print(
    "Customer-experience Tableau datasets "
    "exported successfully."
)

Customer-experience Tableau datasets exported successfully.


In [27]:
capability_columns = [
    col
    for col in channel_capabilities.columns
    if col != "brand"
]

tableau_commercial_access = (
    channel_capabilities
    .melt(
        id_vars="brand",
        value_vars=capability_columns,
        var_name="capability",
        value_name="observed"
    )
)

tableau_commercial_access[
    "observed"
] = (
    tableau_commercial_access[
        "observed"
    ]
    .astype(int)
)

tableau_commercial_access[
    "evidence_status"
] = np.where(
    tableau_commercial_access[
        "observed"
    ] == 1,
    "Observed in collected evidence",
    "Not observed in collected evidence"
)

tableau_commercial_access[
    "interpretation_limit"
] = (
    "Not observed does not establish that "
    "the capability is unavailable"
)

tableau_commercial_access.sort_values(
    [
        "capability",
        "brand"
    ]
)

,brand,capability,observed,evidence_status,interpretation_limit
0,"But First, Coffee",Delivery,1,Observed in collected evidence,Not observed does not establish that the capab...
1,Starbucks,Delivery,1,Observed in collected evidence,Not observed does not establish that the capab...
2,The Coffee Bean & Tea Leaf,Delivery,1,Observed in collected evidence,Not observed does not establish that the capab...
3,"But First, Coffee",Digital Ordering,0,Not observed in collected evidence,Not observed does not establish that the capab...
4,Starbucks,Digital Ordering,1,Observed in collected evidence,Not observed does not establish that the capab...
5,The Coffee Bean & Tea Leaf,Digital Ordering,0,Not observed in collected evidence,Not observed does not establish that the capab...
6,"But First, Coffee",E-commerce,0,Not observed in collected evidence,Not observed does not establish that the capab...
7,Starbucks,E-commerce,0,Not observed in collected evidence,Not observed does not establish that the capab...
8,The Coffee Bean & Tea Leaf,E-commerce,1,Observed in collected evidence,Not observed does not establish that the capab...
9,"But First, Coffee",Gifting,0,Not observed in collected evidence,Not observed does not establish that the capab...


In [28]:
tableau_commercial_access = (
    tableau_commercial_access
    .merge(
        channel_opportunities,
        on="capability",
        how="left"
    )
)

tableau_commercial_access[
    [
        "brand",
        "capability",
        "observed",
        "evidence_status",
        "competitive_evidence",
        "potential_revenue_mechanism",
        "internal_validation_needed",
        "interpretation_limit"
    ]
].sort_values(
    [
        "capability",
        "brand"
    ]
)

,brand,capability,observed,evidence_status,competitive_evidence,potential_revenue_mechanism,internal_validation_needed,interpretation_limit
0,"But First, Coffee",Delivery,1,Observed in collected evidence,NaN,NaN,NaN,Not observed does not establish that the capab...
1,Starbucks,Delivery,1,Observed in collected evidence,NaN,NaN,NaN,Not observed does not establish that the capab...
2,The Coffee Bean & Tea Leaf,Delivery,1,Observed in collected evidence,NaN,NaN,NaN,Not observed does not establish that the capab...
3,"But First, Coffee",Digital Ordering,0,Not observed in collected evidence,Starbucks Mobile Order & Pay observed,"Convenience, frequency and queue reduction","Adoption, order frequency, operational capacit...",Not observed does not establish that the capab...
4,Starbucks,Digital Ordering,1,Observed in collected evidence,Starbucks Mobile Order & Pay observed,"Convenience, frequency and queue reduction","Adoption, order frequency, operational capacit...",Not observed does not establish that the capab...
5,The Coffee Bean & Tea Leaf,Digital Ordering,0,Not observed in collected evidence,Starbucks Mobile Order & Pay observed,"Convenience, frequency and queue reduction","Adoption, order frequency, operational capacit...",Not observed does not establish that the capab...
6,"But First, Coffee",E-commerce,0,Not observed in collected evidence,CBTL marketplace e-commerce observed,Non-store and merchandise revenue,"Demand, fulfillment cost, incremental contribu...",Not observed does not establish that the capab...
7,Starbucks,E-commerce,0,Not observed in collected evidence,CBTL marketplace e-commerce observed,Non-store and merchandise revenue,"Demand, fulfillment cost, incremental contribu...",Not observed does not establish that the capab...
8,The Coffee Bean & Tea Leaf,E-commerce,1,Observed in collected evidence,CBTL marketplace e-commerce observed,Non-store and merchandise revenue,"Demand, fulfillment cost, incremental contribu...",Not observed does not establish that the capab...
9,"But First, Coffee",Gifting,0,Not observed in collected evidence,Starbucks GrabGifts observed,Incremental occasions and prepaid demand,"Incremental occasions, redemption, breakage, m...",Not observed does not establish that the capab...


In [29]:
print(
    "Rows:",
    len(tableau_commercial_access)
)

print(
    "Capabilities:",
    tableau_commercial_access[
        "capability"
    ].nunique()
)

print(
    "Brands:",
    tableau_commercial_access[
        "brand"
    ].nunique()
)

print(
    "Missing opportunity mappings:",
    tableau_commercial_access[
        "potential_revenue_mechanism"
    ].isna().sum()
)

display(
    tableau_commercial_access[
        [
            "brand",
            "capability",
            "observed",
            "evidence_status"
        ]
    ]
    .sort_values(
        [
            "brand",
            "capability"
        ]
    )
)

Rows: 24
Capabilities: 8
Brands: 3
Missing opportunity mappings: 9


,brand,capability,observed,evidence_status
0,"But First, Coffee",Delivery,1,Observed in collected evidence
3,"But First, Coffee",Digital Ordering,0,Not observed in collected evidence
6,"But First, Coffee",E-commerce,0,Not observed in collected evidence
9,"But First, Coffee",Gifting,0,Not observed in collected evidence
12,"But First, Coffee",Loyalty,0,Not observed in collected evidence
15,"But First, Coffee",Payments & Partnerships,1,Observed in collected evidence
18,"But First, Coffee",Physical Access,1,Observed in collected evidence
21,"But First, Coffee",Physical Expansion,1,Observed in collected evidence
1,Starbucks,Delivery,1,Observed in collected evidence
4,Starbucks,Digital Ordering,1,Observed in collected evidence


In [34]:
tableau_revenue_opportunities = (
    revenue_opportunities
    .copy()
)

tableau_revenue_opportunities[
    "opportunity_id"
] = range(
    1,
    len(tableau_revenue_opportunities) + 1
)

tableau_revenue_opportunities[
    "analysis_role"
] = "Management Opportunity"

tableau_revenue_opportunities[
    "financial_status"
] = "Requires internal validation"

tableau_revenue_opportunities[
    "profitability_status"
] = "Actual profitability not estimated"

display(
    tableau_revenue_opportunities[
        [
            "opportunity_id",
            "opportunity",
            "revenue_driver",
            "external_evidence",
            "commercial_hypothesis",
            "recommended_management_test",
            "decision_metric",
            "evidence_status",
            "financial_status",
            "profitability_status"
        ]
    ]
)

,opportunity_id,opportunity,revenue_driver,external_evidence,commercial_hypothesis,recommended_management_test,decision_metric,evidence_status,financial_status,profitability_status
0,1,Promotion Optimization,Revenue efficiency,45/50 observed BFC menu observations carried a...,Some promotions may surrender more revenue tha...,Measure incremental units and contribution mar...,Incremental contribution margin,Strong external signal; internal economics req...,Requires internal validation,Actual profitability not estimated
1,2,Operational Execution,Transaction protection,"Product quality, packaging, value, order execu...","Execution improvements may protect conversion,...","Connect order errors, refunds, remakes and com...",Revenue leakage + repeat-purchase impact,Descriptive signal; internal quantification re...,Requires internal validation,Actual profitability not estimated
2,3,Basket Building,Average Transaction Value,"Collected BFC menu sample is 90% beverage, 6% ...","Food, add-ons, upgrades and bundles may increa...",Test food/add-on attachment and bundles agains...,ATV + contribution margin per transaction,Scenario-supported hypothesis,Requires internal validation,Actual profitability not estimated
3,4,Selective Pricing,Average Selling Price,BFC was lowest-priced across all 5 matched pro...,Selected products may have room for controlled...,Run controlled SKU/branch price tests and meas...,Revenue and contribution after volume response,Statistically supported positioning; test requ...,Requires internal validation,Actual profitability not estimated
4,5,Commercial Access,Transaction Volume,Competitor evidence includes multi-platform de...,Additional access capabilities may create incr...,Pilot selected channels and measure incrementa...,Incremental contribution per channel,Competitive capability hypothesis,Requires internal validation,Actual profitability not estimated


In [35]:
print(
    "Revenue drivers found:"
)

print(
    tableau_revenue_opportunities[
        "revenue_driver"
    ].tolist()
)

Revenue drivers found:
['Revenue efficiency', 'Transaction protection', 'Average Transaction Value', 'Average Selling Price', 'Transaction Volume']


In [39]:
driver_component_map = {
    "Revenue efficiency":
        "Transactions + Average Transaction Value",

    "Transaction protection":
        "Transactions + Retention",

    "Average Transaction Value":
        "Average Transaction Value",

    "Average Selling Price":
        "Average Transaction Value",

    "Transaction Volume":
        "Transactions"
}

tableau_revenue_opportunities[
    "revenue_equation_component"
] = (
    tableau_revenue_opportunities[
        "revenue_driver"
    ]
    .map(driver_component_map)
)

display(
    tableau_revenue_opportunities[
        [
            "opportunity",
            "revenue_driver",
            "revenue_equation_component"
        ]
    ]
)

,opportunity,revenue_driver,revenue_equation_component
0,Promotion Optimization,Revenue efficiency,Transactions + Average Transaction Value
1,Operational Execution,Transaction protection,Transactions + Retention
2,Basket Building,Average Transaction Value,Average Transaction Value
3,Selective Pricing,Average Selling Price,Average Transaction Value
4,Commercial Access,Transaction Volume,Transactions


In [40]:
tableau_internal_data_roadmap = (
    internal_data_roadmap
    .copy()
)

tableau_internal_data_roadmap[
    "purpose"
] = (
    "Internal data required to move "
    "from external opportunity analysis "
    "to measured revenue and profitability analysis"
)

tableau_internal_data_roadmap

,dataset,minimum_fields,revenue_analysis_enabled,purpose
0,POS Transactions,"transaction_id, date, branch, SKU, quantity, s...","ATV, items/transaction, mix, elasticity, branc...",Internal data required to move from external o...
1,Product Cost,"SKU, ingredient cost, packaging cost, variable...",Contribution margin and menu engineering,Internal data required to move from external o...
2,Promotion History,"SKU, campaign, start/end date, discount, exposure",Incrementality and promotion ROI,Internal data required to move from external o...
3,Order Operations,"transaction_id, error, remake, refund, cancell...",Revenue leakage and operational cost,Internal data required to move from external o...
4,Customer / Loyalty,"customer_id, transactions, frequency, basket, ...","Retention, frequency and customer lifetime value",Internal data required to move from external o...
5,Channel Performance,"channel, transactions, revenue, commission, di...",Incremental channel economics,Internal data required to move from external o...


In [41]:
tableau_commercial_access.to_csv(
    TABLEAU / "tableau_commercial_access.csv",
    index=False
)

tableau_revenue_opportunities.to_csv(
    TABLEAU / "tableau_revenue_opportunities.csv",
    index=False
)

tableau_internal_data_roadmap.to_csv(
    TABLEAU / "tableau_internal_data_roadmap.csv",
    index=False
)

print(
    "Commercial-access and revenue-opportunity "
    "datasets exported successfully."
)

Commercial-access and revenue-opportunity datasets exported successfully.


In [42]:
print(
    "Revenue opportunities:",
    len(tableau_revenue_opportunities)
)

print(
    "Missing revenue-equation mappings:",
    tableau_revenue_opportunities[
        "revenue_equation_component"
    ].isna().sum()
)

print(
    "Commercial access rows:",
    len(tableau_commercial_access)
)

print(
    "Internal roadmap rows:",
    len(tableau_internal_data_roadmap)
)

Revenue opportunities: 5
Missing revenue-equation mappings: 0
Commercial access rows: 24
Internal roadmap rows: 6


In [43]:
print(
    "Revenue opportunities:",
    len(tableau_revenue_opportunities)
)

print(
    "Missing revenue-equation mappings:",
    tableau_revenue_opportunities[
        "revenue_equation_component"
    ].isna().sum()
)

display(
    tableau_revenue_opportunities[
        [
            "opportunity",
            "revenue_driver",
            "revenue_equation_component"
        ]
    ]
)

Revenue opportunities: 5
Missing revenue-equation mappings: 0


,opportunity,revenue_driver,revenue_equation_component
0,Promotion Optimization,Revenue efficiency,Transactions + Average Transaction Value
1,Operational Execution,Transaction protection,Transactions + Retention
2,Basket Building,Average Transaction Value,Average Transaction Value
3,Selective Pricing,Average Selling Price,Average Transaction Value
4,Commercial Access,Transaction Volume,Transactions


In [44]:
tableau_files = {
    "Executive KPIs":
        "tableau_executive_kpis.csv",

    "Statistical Evidence":
        "tableau_statistical_evidence.csv",

    "Matched Pricing":
        "tableau_matched_pricing.csv",

    "Pricing Long":
        "tableau_pricing_long.csv",

    "Menu Pricing":
        "tableau_menu_pricing.csv",

    "Revenue Scenarios":
        "tableau_revenue_scenarios.csv",

    "Price Volume Heatmap":
        "tableau_price_volume_heatmap.csv",

    "Promotion Response":
        "tableau_promotion_response.csv",

    "Customer Experience":
        "tableau_customer_experience.csv",

    "Polarity Context":
        "tableau_polarity_context.csv",

    "Customer Methodology":
        "tableau_customer_methodology.csv",

    "Commercial Access":
        "tableau_commercial_access.csv",

    "Revenue Opportunities":
        "tableau_revenue_opportunities.csv",

    "Internal Data Roadmap":
        "tableau_internal_data_roadmap.csv"
}


audit_results = []

for dataset_name, filename in tableau_files.items():

    file_path = TABLEAU / filename

    if file_path.exists():

        df = pd.read_csv(file_path)

        audit_results.append({
            "dataset":
                dataset_name,

            "filename":
                filename,

            "exists":
                True,

            "rows":
                len(df),

            "columns":
                len(df.columns),

            "missing_cells":
                int(
                    df.isna()
                    .sum()
                    .sum()
                ),

            "duplicate_rows":
                int(
                    df.duplicated()
                    .sum()
                )
        })

    else:

        audit_results.append({
            "dataset":
                dataset_name,

            "filename":
                filename,

            "exists":
                False,

            "rows":
                np.nan,

            "columns":
                np.nan,

            "missing_cells":
                np.nan,

            "duplicate_rows":
                np.nan
        })


tableau_audit = pd.DataFrame(
    audit_results
)

display(tableau_audit)

,dataset,filename,exists,rows,columns,missing_cells,duplicate_rows
0,Executive KPIs,tableau_executive_kpis.csv,True,6,5,0,0
1,Statistical Evidence,tableau_statistical_evidence.csv,True,1,8,0,0
2,Matched Pricing,tableau_matched_pricing.csv,True,5,10,0,0
3,Pricing Long,tableau_pricing_long.csv,True,15,5,0,0
4,Menu Pricing,tableau_menu_pricing.csv,True,7,7,0,0
5,Revenue Scenarios,tableau_revenue_scenarios.csv,True,18,8,0,0
6,Price Volume Heatmap,tableau_price_volume_heatmap.csv,True,25,7,0,0
7,Promotion Response,tableau_promotion_response.csv,True,7,7,0,0
8,Customer Experience,tableau_customer_experience.csv,True,21,12,0,0
9,Polarity Context,tableau_polarity_context.csv,True,4,7,0,0


In [45]:
checks = []


def add_check(name, condition, observed, expected):
    checks.append({
        "check":
            name,

        "status":
            "PASS" if condition else "FAIL",

        "observed":
            observed,

        "expected":
            expected
    })


# 1. Matched pricing
pricing_long_check = pd.read_csv(
    TABLEAU / "tableau_pricing_long.csv"
)

add_check(
    "Matched pricing rows",
    len(pricing_long_check) == 15,
    len(pricing_long_check),
    15
)


# 2. Five matched families
family_count = (
    pricing_long_check[
        "product_family"
    ]
    .nunique()
)

add_check(
    "Matched product families",
    family_count == 5,
    family_count,
    5
)


# 3. BFC lowest-ranked in every matched family
bfc_ranks = (
    pricing_long_check[
        pricing_long_check["brand"]
        == "But First, Coffee"
    ]["price_rank"]
)

bfc_all_rank_1 = (
    bfc_ranks.eq(1).all()
)

add_check(
    "BFC lowest price rank",
    bfc_all_rank_1,
    bfc_ranks.tolist(),
    "Rank 1 in all 5 families"
)


# 4. Revenue scenarios
scenario_check = pd.read_csv(
    TABLEAU / "tableau_revenue_scenarios.csv"
)

add_check(
    "Revenue scenario rows",
    len(scenario_check) == 18,
    len(scenario_check),
    18
)


# 5. Promotion neutrality
promo_check = pd.read_csv(
    TABLEAU / "tableau_promotion_response.csv"
)

neutral_row = promo_check[
    promo_check[
        "volume_increase_pct"
    ] == 25
]

neutral_value = (
    neutral_row[
        "revenue_change_pct"
    ].iloc[0]
)

add_check(
    "20% discount revenue neutrality",
    np.isclose(
        neutral_value,
        0
    ),
    neutral_value,
    0
)


# 6. Customer experience rows
customer_check = pd.read_csv(
    TABLEAU / "tableau_customer_experience.csv"
)

add_check(
    "Customer experience rows",
    len(customer_check) == 21,
    len(customer_check),
    21
)


# 7. BFC review base
review_bases = (
    customer_check[
        "review_base_n"
    ]
    .unique()
    .tolist()
)

add_check(
    "BFC review base",
    review_bases == [104],
    review_bases,
    [104]
)


# 8. Commercial access
commercial_check = pd.read_csv(
    TABLEAU / "tableau_commercial_access.csv"
)

add_check(
    "Commercial access rows",
    len(commercial_check) == 24,
    len(commercial_check),
    24
)


# 9. Revenue opportunities
opportunity_check = pd.read_csv(
    TABLEAU / "tableau_revenue_opportunities.csv"
)

add_check(
    "Revenue opportunities",
    len(opportunity_check) == 5,
    len(opportunity_check),
    5
)


# 10. Revenue-equation mapping
missing_driver_mapping = (
    opportunity_check[
        "revenue_equation_component"
    ]
    .isna()
    .sum()
)

add_check(
    "Revenue equation mappings",
    missing_driver_mapping == 0,
    missing_driver_mapping,
    0
)


integrity_checks = pd.DataFrame(
    checks
)

display(integrity_checks)

,check,status,observed,expected
0,Matched pricing rows,PASS,15,15
1,Matched product families,PASS,5,5
2,BFC lowest price rank,PASS,"[1.0, 1.0, 1.0, 1.0, 1.0]",Rank 1 in all 5 families
3,Revenue scenario rows,PASS,18,18
4,20% discount revenue neutrality,PASS,0.0,0
5,Customer experience rows,PASS,21,21
6,BFC review base,PASS,[104],[104]
7,Commercial access rows,PASS,24,24
8,Revenue opportunities,PASS,5,5
9,Revenue equation mappings,PASS,0,0


In [46]:
gap_vs_starbucks_pct = (
    -pricing_headroom[
        "bfc_discount_vs_starbucks_pct"
    ].mean()
)

gap_vs_cbtl_pct = (
    -pricing_headroom[
        "bfc_discount_vs_cbtl_pct"
    ].mean()
)

print(
    "Mean matched-family gap vs Starbucks:",
    round(gap_vs_starbucks_pct, 2)
)

print(
    "Mean matched-family gap vs CBTL:",
    round(gap_vs_cbtl_pct, 2)
)

Mean matched-family gap vs Starbucks: -28.23
Mean matched-family gap vs CBTL: -34.14


In [48]:
# Reload the existing Tableau KPI dataset
tableau_executive_kpis = pd.read_csv(
    TABLEAU / "tableau_executive_kpis.csv"
)

# Apply the reconciled mean matched-family price gaps
tableau_executive_kpis.loc[
    tableau_executive_kpis["kpi"]
    == "Matched Price Gap vs Starbucks",
    "value"
] = gap_vs_starbucks_pct

tableau_executive_kpis.loc[
    tableau_executive_kpis["kpi"]
    == "Matched Price Gap vs CBTL",
    "value"
] = gap_vs_cbtl_pct

# Export the corrected dataset
tableau_executive_kpis.to_csv(
    TABLEAU / "tableau_executive_kpis.csv",
    index=False
)

display(tableau_executive_kpis)

,kpi,value,unit,display_direction,evidence_type
0,BFC Beverage Median Price,130.000000,PHP,Neutral,Descriptive
1,Matched Price Gap vs Starbucks,-28.227332,Percent,Lower,Matched-price descriptive
2,Matched Price Gap vs CBTL,-34.141782,Percent,Lower,Matched-price descriptive
3,Observed Promotion Coverage,90.000000,Percent,Observed,Observed snapshot
4,Observed Promotion Discount,20.000000,Percent,Observed,Observed snapshot
5,Volume Lift for 20% Discount Neutrality,25.000000,Percent,Required,Scenario calculation


In [49]:
print(
    tableau_executive_kpis[
        ["kpi", "value"]
    ]
)

                                       kpi       value
0                BFC Beverage Median Price  130.000000
1           Matched Price Gap vs Starbucks  -28.227332
2                Matched Price Gap vs CBTL  -34.141782
3              Observed Promotion Coverage   90.000000
4              Observed Promotion Discount   20.000000
5  Volume Lift for 20% Discount Neutrality   25.000000
